In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.cm as mcm
from matplotlib.colors import Normalize
from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from shapely.geometry import box as shapely_box
from pyproj import Transformer
from adjustText import adjust_text

# ── 0. Paths & settings ───────────────────────────────────────────────────────
FLOW_DIR   = Path("/intercity_flow")
LOCAL_DIR  = Path("/city_patient_sum")
SHP_PATH   = Path("/city_shp/shi_en.shp")
CHINA_SHP  = Path("/中国底图-中图社审过版本/中国底图/中国面.shp")
CHINA_SHP2 = Path("/中国国界线/九段线/九段线和群岛.shp")
OUTFILE    = Path("/FigS5.png")
SCENARIO   = "earlypeak_NZ_CL"
YEARS      = [2020, 2030, 2040, 2050, 2060]

CITY_NAME_MAP = {
    "Wulumuqi":  "Urumqi",
    "Xian":      "Xi'an",
    "Qiqihaer":  "Qiqihar",
    "Huhehaote": "Hohhot",
    "Haerbin":   "Harbin",
}
PROJ_STR = "+proj=aea +lat_1=25 +lat_2=47 +lat_0=0 +lon_0=105"

# ── 1. Global style ───────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family": "Arial",
    "font.size":   6,
})

# ── 2. Spatial data ───────────────────────────────────────────────────────────
china_border = gpd.read_file(CHINA_SHP).to_crs(PROJ_STR)
jiudanline   = gpd.read_file(CHINA_SHP2).to_crs(PROJ_STR)

city_shp_raw = gpd.read_file(SHP_PATH)
city_shp_raw["English"] = city_shp_raw["English"].str.strip().map(
    lambda x: CITY_NAME_MAP.get(x, x))
city_shp = city_shp_raw.to_crs(PROJ_STR)

_hhy_transformer = Transformer.from_crs("EPSG:4326", PROJ_STR, always_xy=True)
_HHY_X, _HHY_Y  = _hhy_transformer.transform([127.5, 98.5], [50.2, 25.0])

_NANHAI_BOUNDS = (
    gpd.GeoDataFrame(geometry=[shapely_box(105, 2, 122, 24)], crs="EPSG:4326")
    .to_crs(PROJ_STR).total_bounds
)

_hhy_x0, _hhy_y0 = _HHY_X[0], _HHY_Y[0]
_hhy_x1, _hhy_y1 = _HHY_X[1], _HHY_Y[1]

def _hhy_x_at_y(y):
    t = (y - _hhy_y1) / (_hhy_y0 - _hhy_y1)
    return _hhy_x1 + t * (_hhy_x0 - _hhy_x1)

xmin_c, ymin_c, xmax_c, ymax_c = china_border.total_bounds

# ── 3. Data loaders ───────────────────────────────────────────────────────────
def rename_idx(idx):
    return idx.str.strip().map(lambda x: CITY_NAME_MAP.get(x, x))

def load_flow_matrix(year):
    path = FLOW_DIR / f"flow_patientnum_avg_{SCENARIO}_{year}.csv"
    df   = pd.read_csv(path, index_col=0)
    df.index   = rename_idx(df.index)
    df.columns = rename_idx(df.columns)
    df = df.loc[~df.index.isin(["total"]), ~df.columns.isin(["total"])]
    np.fill_diagonal(df.values, 0)
    return df

def load_citysum(year):
    path = LOCAL_DIR / f"citysum_{SCENARIO}_{year}.csv"
    df   = pd.read_csv(path)
    df["city"] = df["city"].str.strip().map(lambda x: CITY_NAME_MAP.get(x, x))
    return df.groupby("city")[["local_patient", "mo_total"]].sum()

def compute_year(year):
    df_flow  = load_flow_matrix(year)
    df_local = load_citysum(year)
    inflow   = df_flow.sum(axis=0).groupby(level=0).sum()
    outflow  = df_flow.sum(axis=1).groupby(level=0).sum()
    all_cities = inflow.index.union(outflow.index)
    inflow   = inflow.reindex(all_cities, fill_value=0)
    outflow  = outflow.reindex(all_cities, fill_value=0)
    net      = (inflow ).rename("net")
    df_local = df_local.groupby(level=0).sum()
    common   = net.index.intersection(df_local.index)
    out      = df_local.loc[common].copy()
    out["net"]    = net.loc[common].values
    out["demand"] = out["net"] + out["local_patient"]
    return out

# ── 4. Pre-compute all years ──────────────────────────────────────────────────
year_data = {}
for y in YEARS:
    print(f"Loading {y}...")
    yr = compute_year(y).reset_index().rename(
        columns={"index": "city"})
    yr = yr.groupby("city", as_index=False)[
        ["mo_total", "demand", "net"]].sum()
    year_data[y] = yr

# ── 5. Shared color norm (across all years & both cols) ───────────────────────
all_vals = []
for y in YEARS:
    all_vals.extend(year_data[y]["mo_total"].dropna().tolist())
    all_vals.extend(year_data[y]["demand"].dropna().tolist())
all_vals = np.array(all_vals)

vmin_map = np.percentile(all_vals, 1)
vmax_map = np.percentile(all_vals, 99)
CMAP     = "YlOrRd"
norm     = Normalize(vmin=vmin_map, vmax=vmax_map)

# ── 6. Nanhai inset helper ────────────────────────────────────────────────────
def add_nanhai(parent_ax, shp_yr, col):
    x1, y1, x2, y2 = _NANHAI_BOUNDS
    axins = parent_ax.inset_axes([0.82, 0.01, 0.16, 0.20])
    axins.set_facecolor("white")
    shp_yr.plot(column=col, ax=axins, cmap=CMAP, norm=norm,
                linewidth=0, missing_kwds={"color": "lightgrey"})
    china_border.plot(ax=axins, facecolor="none",
                      edgecolor="black", linewidth=0.2)
    jiudanline.plot(ax=axins, edgecolor="black", linewidth=0.4)
    axins.set_xlim(x1, x2)
    axins.set_ylim(y1, y2)
    axins.tick_params(left=False, bottom=False,
                      labelleft=False, labelbottom=False)
    for sp in axins.spines.values():
        sp.set_linewidth(0.5)
        sp.set_color("black")

# ── 7. Single map drawing function ───────────────────────────────────────────
PANEL_TAGS = [
    ["a", "b", "c", "d", "e"],   # row 0: mo_total
    ["f", "g", "h", "i", "j"],   # row 1: demand
]
ROW_LABELS = ["Before mobility", "After mobility"]
COL_KEYS   = ["mo_total", "demand"]

def draw_one(ax, shp_yr, col, year, tag, show_row_label=False):
    shp_yr.plot(
        column=col, ax=ax,
        cmap=CMAP, norm=norm,
        linewidth=0.15, edgecolor="#AAAAAA",
        missing_kwds={"color": "#DDDDDD"},
    )
    china_border.plot(ax=ax, facecolor="none",
                      edgecolor="black", linewidth=0.15)
    jiudanline.plot(ax=ax, edgecolor="black", linewidth=0.25)

    # Hu line
    ax.plot(_HHY_X, _HHY_Y, color="black", lw=0.6,
            ls="--", dashes=(4, 3), zorder=5)

    # Map extent
    height = ymax_c - ymin_c
    ax.set_xlim(xmin_c, xmax_c)
    ax.set_ylim(ymin_c + height * 0.10, ymax_c)
    ax.set_axis_off()

    # Panel tag
    ax.text(-0.04, 1.01, tag, transform=ax.transAxes,
            fontsize=7, fontweight="bold", va="bottom")

    # Year title (col header, only row 0)
    if show_row_label:
        ax.set_title(str(year), fontsize=6, fontweight="bold",
                     pad=2, loc="center")

    # Top 5 cities
    top5 = (
        shp_yr.dropna(subset=[col])
        .nlargest(5, col)
    )
    texts = []
    for _, row in top5.iterrows():
        cx_r = row.geometry.centroid.x
        cy_r = row.geometry.centroid.y
        val  = row[col]
        txt  = ax.text(
            cx_r, cy_r,
            f"{row['English']}\n({val/1000:.1f}k)",
            fontsize=3.8, ha="center", va="center",
            color="#1A1A1A", zorder=8,
            bbox=dict(boxstyle="round,pad=0.12",
                      fc="white", ec="none", alpha=0.65),
        )
        texts.append(txt)
    if texts:
        adjust_text(
            texts, ax=ax,
            time_lim=1,
            arrowprops=dict(arrowstyle="-",
                            color="#555555", lw=0.35),
        )

    add_nanhai(ax, shp_yr, col)

# ── 8. Layout: 2 rows × 5 cols ───────────────────────────────────────────────
W_IN = 18 / 2.54
H_IN = 12 / 2.54

fig = plt.figure(figsize=(W_IN, H_IN), dpi=400, facecolor="white")
gs  = GridSpec(
    2, 5,
    figure=fig,
    hspace=0.08,
    wspace=0.04,
    left=0.04, right=0.96,
    top=0.93,  bottom=0.08,
)

axes = [[fig.add_subplot(gs[r, c]) for c in range(5)] for r in range(2)]

# ── 9. Draw all panels ────────────────────────────────────────────────────────
for ci, year in enumerate(YEARS):
    yr   = year_data[year]
    shp_yr = city_shp.merge(yr, left_on="English",
                             right_on="city", how="left")

    for ri, col in enumerate(COL_KEYS):
        tag = PANEL_TAGS[ri][ci]
        draw_one(
            ax=axes[ri][ci],
            shp_yr=shp_yr,
            col=col,
            year=year,
            tag=tag,
            show_row_label=(ri == 0),   # year title only on top row
        )

# ── 10. Row labels (left side) ────────────────────────────────────────────────
for ri, label in enumerate(ROW_LABELS):
    axes[ri][0].text(
        -0.08, 0.5, label,
        transform=axes[ri][0].transAxes,
        fontsize=6, fontweight="bold",
        va="center", ha="right",
        rotation=90,
    )

# ── 11. Shared colorbar ───────────────────────────────────────────────────────
axes_flat = [ax for row in axes for ax in row]

sm = mcm.ScalarMappable(cmap=CMAP, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm,
    ax=axes_flat,          
    orientation="horizontal",
    fraction=0.018,
    pad=0.02,
    shrink=0.35,
    aspect=25,
)
cbar.ax.tick_params(labelsize=5.5)
cbar.set_label("Patients per year", fontsize=6)

# ── 12. Save ──────────────────────────────────────────────────────────────────
OUTFILE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTFILE, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"✓ Saved → {OUTFILE}")

DataSourceError: /中国底图-中图社审过版本/中国底图/中国面.shp: No such file or directory